In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SLUG = "pensioni_pa_dag_2024"
MART_TABLE = "mart_regioni_dicembre"
METRICA = "numero_partite"
DIM = "regione"

cwd = Path.cwd().resolve()
if (cwd / "dataset.yml").exists():
    candidate_dir = cwd
elif (cwd.parent / "dataset.yml").exists():
    candidate_dir = cwd.parent
else:
    raise FileNotFoundError(f"Dataset dir non trovato da cwd={cwd}")

MART_PATH = candidate_dir.parents[1] / "out" / "data" / "mart" / SLUG / "2024"
if not MART_PATH.exists():
    raise FileNotFoundError(f"Mart non trovato: {MART_PATH}. Eseguire prima toolkit run all.")

PARQUET_PATH = MART_PATH / f"{MART_TABLE}.parquet"
if not PARQUET_PATH.exists():
    raise FileNotFoundError(f"Tabella mart non trovata: {PARQUET_PATH}")

PARQUET_PATH

In [ ]:
con = duckdb.connect()
df = con.execute("SELECT * FROM read_parquet(?)", [str(PARQUET_PATH)]).df()
print(f"Shape: {df.shape}")
display(df.dtypes)

In [ ]:
display(df.head(10))

In [ ]:
print("Null per colonna:")
display(df.isnull().sum())

if METRICA in df.columns:
    print(f"\nRange {METRICA}: min={df[METRICA].min()}, max={df[METRICA].max()}, negativi={(df[METRICA] < 0).sum()}")
else:
    raise KeyError(f"Colonna metrica non trovata: {METRICA}")

In [ ]:
(
    df.groupby(DIM)[METRICA]
    .sum()
    .sort_values(ascending=False)
    .plot(kind="bar", figsize=(12, 5), title=f"{METRICA} per {DIM}")
)
plt.tight_layout()
plt.show()

## Note v0

- Slug: `pensioni_pa_dag_2024` | Tabella mart: `mart_regioni_dicembre`
- Metrica guida: `numero_partite`
- Perimetro: snapshot regionale di dicembre 2024, solo `DIRETTA` e `INDIRETTA/REVERSIBILE`
- Questo notebook e' esplorativo e resta in DI, non e' l'analisi pubblica.